[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-3/lab-3.3-backward-pass.ipynb)

# LAB·3.3 · The backward pass

**Hardware:** correctness anywhere; timing on TPU.

A fast forward with a slow backward is worthless for training. Paper first, again: derive dQ, dK, dV for softmax attention, find the D = rowsum(dO ⊙ O) identity that collapses the softmax Jacobian, and see why the backward *recomputes* P blockwise instead of storing it. Then wire your forward to a streaming backward with `custom_vjp` and let `jax.grad` referee.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
# reference math and the custom_vjp pair; forward saves only (O, m, l) style residuals
def attention_ref(q, k, v):
    return jax.nn.softmax(q @ k.T, axis=-1) @ v

@jax.custom_vjp
def attention(q, k, v):
    return attention_ref(q, k, v)

def fwd(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    l = jnp.sum(p, axis=-1, keepdims=True)
    o = (p / l) @ v
    return o, (q, k, v, o, m, l)

def bwd(res, do):
    q, k, v, o, m, l = res
    # recompute P blockwise in the real kernel; whole-array here for clarity
    p = jnp.exp(q @ k.T - m) / l
    dv = p.T @ do
    d = jnp.sum(do * o, axis=-1, keepdims=True)   # the identity that kills the Jacobian
    dp = do @ v.T
    ds = p * (dp - d)
    return ds @ k, ds.T @ q, dv

attention.defvjp(fwd, bwd)

In [ ]:
q = jax.random.normal(jax.random.key(0), (128, 64))
k = jax.random.normal(jax.random.key(1), (256, 64))
v = jax.random.normal(jax.random.key(2), (256, 64))

def loss_custom(q, k, v): return jnp.sum(jnp.tanh(attention(q, k, v)))
def loss_ref(q, k, v): return jnp.sum(jnp.tanh(attention_ref(q, k, v)))

g1 = jax.grad(loss_custom, argnums=(0, 1, 2))(q, k, v)
g2 = jax.grad(loss_ref, argnums=(0, 1, 2))(q, k, v)
for name, a, b in zip("qkv", g1, g2):
    check(f"d{name}", a, b, tol=1e-5)

## The streaming upgrade

Everything above is whole-array for clarity. The exercise that earns the gate: re-express `bwd` as a blocked streaming pass (same fori_loop shape as LAB·3.2, with P recomputed per block from the saved (m, l)), then port it into a Pallas kernel. Differential-test at every step: values 1e-3, grads 1e-2 in bf16, corner cases included (a fully-masked row, a length-1 sequence). The gate table on the site holds these exact tolerances.